In [ ]:
import cv2
import numpy as np
import os
import time
from ultralytics import YOLO

# Configuration
frameWidth = 640
frameHeight = 480
cameraFeed = False  # Set to True to use camera feed
cameraNo = 1
videoPath = r"C:\Users\dai\PycharmProjects\NLP-CV\S12\images\robots.mp4"  # Specify your video file path
outputDir = r"C:\Users\dai\PycharmProjects\NLP-CV\S12\images"  # Specify your output directory
modelPath = "yolov5x.pt"  # Use the proper YOLOv5x model identifier
labelMapPath = "C:/Users/arnav/OneDrive/Documents/Internal Internship/coco.names"  # Path to class names file

# Load class names
with open(labelMapPath, "r") as f:
    classes = [line.strip() for line in f.readlines()]
colors = np.random.uniform(0, 255, size=(len(classes), 3))

# Load YOLOv5 model
try:
    model = YOLO(modelPath)  # This will automatically download 'yolov5x.pt' if not already downloaded
except Exception as e:
    raise RuntimeError(f"Failed to load YOLO model: {e}")

# Initialize video capture
cap = cv2.VideoCapture(cameraNo if cameraFeed else videoPath)
if not cap.isOpened():
    raise ValueError(f"Cannot open video source: {videoPath if not cameraFeed else cameraNo}")

if not os.path.exists(outputDir):
    raise ValueError(f"Output directory does not exist: {outputDir}")

output_file = os.path.join(outputDir, os.path.basename(videoPath).replace('.mp4', '_Detection.avi')
                            if not cameraFeed else 'camera_feed_detection.avi')

# Get FPS and frame size
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0.0:
    fps = 25.0  # Default fallback FPS

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_size = (width, height)

# Initialize VideoWriter
fourcc = cv2.VideoWriter_fourcc(*'XVID')  # Use a supported codec
video_writer = cv2.VideoWriter(output_file, fourcc, fps, frame_size)

if not video_writer.isOpened():
    raise ValueError(f"Failed to initialize VideoWriter with file: {output_file}")

starting_time = time.time()
frame_id = 0

# Processing loop
while True:
    success, img = cap.read()
    if not success:
        print('[i] ==> Done processing!!!')
        print('[i] ==> Output file is stored at', output_file)
        break

    frame_id += 1

    # Resize the frame for consistent processing
    resized_img = cv2.resize(img, (frameWidth, frameHeight))

    # Perform detection using YOLOv5x
    results = model(resized_img)  # Perform inference

    # Parse detections
    for result in results:  # Iterate through detections
        for box in result.boxes:
            x_min, y_min, x_max, y_max = map(int, box.xyxy[0].tolist())  # Get bounding box coordinates
            confidence = box.conf[0]  # Get confidence score
            class_id = int(box.cls[0])  # Get class ID

            if confidence > 0.65:  # Confidence threshold
                label = f"{classes[class_id]}: {confidence:.2f}%"
                color = colors[class_id]
                cv2.rectangle(resized_img, (x_min, y_min), (x_max, y_max), color, 2)
                cv2.putText(resized_img, label, (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Display FPS
    elapsed_time = time.time() - starting_time
    fps = frame_id / elapsed_time
    cv2.putText(resized_img, f"FPS: {fps:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # Show image
    cv2.imshow("YOLOv5 Detection", resized_img)
    video_writer.write(resized_img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
video_writer.release()
cv2.destroyAllWindows()
print('==> All done!')

PRO TIP  Replace 'model=yolov5x.pt' with new 'model=yolov5xu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.


0: 480x640 7 persons, 2 cars, 4 motorcycles, 1 bus, 2 trucks, 3 umbrellas, 2 handbags, 80.4ms
Speed: 5.0ms preprocess, 80.4ms inference, 195.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 7 persons, 2 cars, 4 motorcycles, 1 bus, 2 trucks, 3 umbrellas, 2 handbags, 63.8ms
Speed: 2.0ms preprocess, 63.8ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 6 persons, 2 cars, 4 motorcycles, 1 bus, 1 truck, 3 umbrellas, 64.1ms
Speed: 1.0ms preprocess, 64.1ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 5 persons, 3 cars, 4 motorcycles, 1 bus, 2 trucks, 3 umbrellas, 63.3ms
Speed: 1.5ms preprocess, 63.3ms inference, 4.0ms postprocess per image at shape (1, 3, 480, 640)